In [1]:
# Cell-1

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats import multitest
import re
import warnings
import os
from itertools import combinations

# Suppress specific warnings that might clutter output
warnings.filterwarnings("ignore", message="The default of the `n_jobs` argument in `fast_power_transform` has been deprecated and will be changed to 1 in a future release.")
warnings.filterwarnings("ignore", category=UserWarning, module='statsmodels')

# --- Function to clean column names for statsmodels formula ---
def clean_column_name(name):
    """
    Cleans a column name to be compatible with statsmodels formulas.
    """
    cleaned_name = re.sub(r'[^a-zA-Z0-9_]', '_', name)
    cleaned_name = re.sub(r'_{2,}', '_', cleaned_name)
    cleaned_name = cleaned_name.strip('_')
    if cleaned_name and cleaned_name[0].isdigit():
        cleaned_name = '_' + cleaned_name
    return cleaned_name

# --- Define file paths ---
genetic_data_path = 'genotype_matrix.csv'
connectivity_data_path = 'Connectivity_matrix_all_subjects_region_pairs.csv'
covariates_data_path = 'subject_covariates.csv'
snp_metadata_path = 'snp_metadata.csv'

output_folder = 'Output_Version2'
os.makedirs(output_folder, exist_ok=True)

print("--- Starting Data Loading and Preprocessing ---")

# --- Step 1: Data Loading ---
try:
    genetic_data = pd.read_csv(genetic_data_path)
    connectivity_data = pd.read_csv(connectivity_data_path)
    covariates_data = pd.read_csv(covariates_data_path)
    snp_metadata = pd.read_csv(snp_metadata_path)
    print("All data files loaded successfully.")
except FileNotFoundError as e:
    print(f"Error: {e}. Please ensure all files are in the correct directory.")
    exit()

--- Starting Data Loading and Preprocessing ---
All data files loaded successfully.


In [6]:
# Cell-2

# Apply Fisher's r-to-z transformation to connectivity data
connectivity_cols_to_transform = [col for col in connectivity_data.columns if connectivity_data[col].dtype == 'float64']
for col in connectivity_cols_to_transform:
    connectivity_data[col] = np.arctanh(np.clip(connectivity_data[col], -0.999999, 0.999999))
print("Fisher's r-to-z transformation applied to connectivity data.")

# Harmonize Subject ID column names
if 'Subject ID' in connectivity_data.columns:
    connectivity_data.rename(columns={'Subject ID': 'Subject_ID'}, inplace=True)
if 'subject_id' in covariates_data.columns:
    covariates_data.rename(columns={'subject_id': 'Subject_ID'}, inplace=True)
if 'SubjectId' in genetic_data.columns:
    genetic_data.rename(columns={'SubjectId': 'Subject_ID'}, inplace=True)
print("Subject ID column names harmonized.")

# --- Filter out the specified subject ---
subject_to_exclude = '029_S_2395'
genetic_data = genetic_data[genetic_data['Subject_ID'] != subject_to_exclude]
connectivity_data = connectivity_data[connectivity_data['Subject_ID'] != subject_to_exclude]
covariates_data = covariates_data[covariates_data['Subject_ID'] != subject_to_exclude]
print(f"Subject '{subject_to_exclude}' has been excluded from all datasets.")

# --- Step 2: Merging Data for Analysis ---
merged_data = pd.merge(genetic_data, covariates_data, on='Subject_ID', how='inner')
merged_data = pd.merge(merged_data, connectivity_data, on='Subject_ID', how='inner')
print(f"Dataframes merged successfully. Total subjects: {len(merged_data)}")

# --- Preprocessing for model fitting ---
if 'group' in merged_data.columns:
    merged_data.rename(columns={'group': 'Subtype'}, inplace=True)

# Clean connectivity column names to avoid formula errors
original_connectivity_columns_in_df = [col for col in connectivity_data.columns if col != 'Subject_ID']
cleaned_connectivity_name_map = {col: clean_column_name(col) for col in original_connectivity_columns_in_df}
merged_data.rename(columns=cleaned_connectivity_name_map, inplace=True)
print("Connectivity column names cleaned.")

# Convert categorical variables and set Control as the new reference
merged_data['Subtype'] = pd.Categorical(merged_data['Subtype'], categories=['Control', 'TypAD', 'AsymAD', 'LowNFT'], ordered=False)
merged_data['scanner'] = pd.Categorical(merged_data['scanner'])
merged_data['scan_type'] = pd.Categorical(merged_data['scan_type'])

print("\nData prepared for model fitting.")

Fisher's r-to-z transformation applied to connectivity data.
Subject ID column names harmonized.
Subject '029_S_2395' has been excluded from all datasets.
Dataframes merged successfully. Total subjects: 187
Connectivity column names cleaned.

Data prepared for model fitting.


In [7]:
# Cell-3

print("\n--- Starting Full Automated Regression Pipeline ---")

# --- Step 3: Preprocessing and Feature Identification ---
snp_columns = [col for col in merged_data.columns if col.startswith('rs')]
connectivity_columns = [col for col in merged_data.columns if 'Connectivity' in col]

snp_metadata['comparison_group'] = snp_metadata.apply(lambda row: '_vs_'.join(sorted([str(row['Case']), str(row['Control'])])), axis=1)
gwas_snp_groups = {}
unique_comparison_groups = snp_metadata['comparison_group'].unique()
for group_name in unique_comparison_groups:
    rs_ids_for_group = snp_metadata[snp_metadata['comparison_group'] == group_name]['SNP'].tolist()
    actual_snp_cols_in_data = [
        col for col in snp_columns
        if any(col.startswith(rs_id + '_') for rs_id in rs_ids_for_group)
    ]
    gwas_snp_groups[group_name] = actual_snp_cols_in_data

# --- Step 4: OLS Regression Loop ---
all_model_results = []
MIN_OBS_FOR_MODEL = 10
model_formula_template = "{connectivity_col} ~ {snp_col} * C(Subtype, Treatment('Control')) + age + sex + C(scanner) + C(scan_type)"

pairwise_subtypes = ['AsymAD', 'TypAD', 'LowNFT']
pairwise_combinations = list(combinations(pairwise_subtypes, 2))

# Flatten the list of all SNPs to loop through them
all_gwas_snps = {snp for snps in gwas_snp_groups.values() for snp in snps}

processed_count = 0
total_combinations = len(all_gwas_snps) * len(connectivity_columns)

for snp_col in all_gwas_snps:
    for connectivity_col in connectivity_columns:
        processed_count += 1
        
        # Select and clean the subset of data for this model
        cols_for_model = [connectivity_col, snp_col, 'Subtype', 'age', 'sex', 'scanner', 'scan_type']
        model_data_subset = merged_data[cols_for_model].dropna()

        # Check for model feasibility
        if len(model_data_subset) < MIN_OBS_FOR_MODEL:
            continue
        if model_data_subset[snp_col].std() == 0 or model_data_subset[connectivity_col].std() == 0:
            continue
        if len(model_data_subset['Subtype'].dropna().unique()) < 2:
            continue

        result_row = {
            'SNP_Name': snp_col,
            'Connectivity_Name': connectivity_col,
            'R_squared': np.nan,
        }
        
        # Determine GWAS group
        gwas_comp = [name for name, snps in gwas_snp_groups.items() if snp_col in snps]
        result_row['GWAS_Comparison'] = gwas_comp[0] if gwas_comp else 'N/A'

        try:
            model = smf.ols(model_formula_template.format(connectivity_col=connectivity_col, snp_col=snp_col), data=model_data_subset).fit()
            param_names = model.params.index.tolist()
            result_row['R_squared'] = model.rsquared

            # --- Get results for contrasts against the reference group (Control) ---
            for subtype in pairwise_subtypes:
                param_name = f"{snp_col}:C(Subtype, Treatment('Control'))[T.{subtype}]"
                if param_name in param_names:
                    result_row[f'P_{subtype}_vs_Control'] = model.pvalues[param_name]
                    result_row[f'Coeff_{subtype}_vs_Control'] = model.params[param_name]

            # --- Get results for custom contrasts (non-reference groups) ---
            for group1, group2 in pairwise_combinations:
                param1_name = f"{snp_col}:C(Subtype, Treatment('Control'))[T.{group1}]"
                param2_name = f"{snp_col}:C(Subtype, Treatment('Control'))[T.{group2}]"

                if param1_name in param_names and param2_name in param_names:
                    contrast_matrix = np.zeros(len(param_names))
                    contrast_matrix[param_names.index(param1_name)] = 1
                    contrast_matrix[param_names.index(param2_name)] = -1
                    
                    test_result = model.t_test(contrast_matrix)
                    p_value = test_result.pvalue.item() if isinstance(test_result.pvalue, np.ndarray) else test_result.pvalue
                    coefficient = test_result.effect.item() if isinstance(test_result.effect, np.ndarray) else test_result.effect

                    result_row[f'P_{group1}_vs_{group2}'] = p_value
                    result_row[f'Coeff_{group1}_vs_{group2}'] = coefficient
            
            all_model_results.append(result_row)
            
        except Exception:
            # Skip models that fail to converge or have other issues
            pass

        if processed_count % 100000 == 0:
            print(f"Processed {processed_count}/{total_combinations} combinations...")


# --- Step 5: Save All Raw Results ---
results_df = pd.DataFrame(all_model_results)
results_df.to_csv(os.path.join(output_folder, 'all_regression_result.csv'), index=False)
print("\n'all_regression_result.csv' saved.")

# --- Step 6: Apply FDR Correction ---
p_value_cols = [col for col in results_df.columns if col.startswith('P_') and '_vs_' in col]
corrected_df = results_df.copy()

for p_col in p_value_cols:
    valid_p_values = corrected_df[p_col].dropna()
    original_indices = valid_p_values.index
    if len(valid_p_values) > 0:
        reject, pvals_corrected, _, _ = multitest.multipletests(pvals=valid_p_values, alpha=0.05, method='fdr_bh')
        corrected_df.loc[original_indices, f'P_FDR{p_col[1:]}'] = pvals_corrected

corrected_df.to_csv(os.path.join(output_folder, 'all_regression_result_FDR.csv'), index=False)
print("'all_regression_result_FDR.csv' saved.")

# --- Step 7: Filter for Significant Results ---
fdr_cols = [col for col in corrected_df.columns if col.startswith('P_FDR')]
combined_mask = pd.DataFrame(False, index=corrected_df.index, columns=[1])
for fdr_col in fdr_cols:
    if fdr_col in corrected_df.columns:
        mask = corrected_df[fdr_col] < 0.05
        combined_mask = combined_mask | mask.to_frame()
        
significant_results = corrected_df[combined_mask.iloc[:,0]].copy()

significant_results.to_csv(os.path.join(output_folder, 'significant_result.csv'), index=False)
print("'significant_result.csv' saved.")

print("\n--- Pipeline Complete. ---")


--- Starting Full Automated Regression Pipeline ---
Processed 100000/4011033 combinations...
Processed 200000/4011033 combinations...
Processed 300000/4011033 combinations...
Processed 400000/4011033 combinations...
Processed 500000/4011033 combinations...
Processed 600000/4011033 combinations...
Processed 700000/4011033 combinations...
Processed 800000/4011033 combinations...
Processed 900000/4011033 combinations...
Processed 1000000/4011033 combinations...
Processed 1100000/4011033 combinations...
Processed 1200000/4011033 combinations...
Processed 1300000/4011033 combinations...
Processed 1400000/4011033 combinations...
Processed 1500000/4011033 combinations...
Processed 1600000/4011033 combinations...
Processed 1700000/4011033 combinations...
Processed 1800000/4011033 combinations...
Processed 1900000/4011033 combinations...
Processed 2000000/4011033 combinations...
Processed 2100000/4011033 combinations...
Processed 2200000/4011033 combinations...
Processed 2300000/4011033 combin